# Deep Learning Practice 

In [46]:
import pandas as pd
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [47]:
#drop the irrevelent columns
data=data.drop(
    columns=['RowNumber','CustomerId','Surname']
)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [48]:
"""Encoding the Categorical data like the Gender: 
Male- 1, Female- 0, 
we use the label encoder as this has only 2 lables:
1. Male 2. female
"""
from sklearn.preprocessing import LabelEncoder
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [49]:
"""
Since now there are 3 different cities, we use OHE to categorize 
them into a matrix. [[1,0,0],[0,0,1]],etc.
"""
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo=onehot_encoder_geo.fit_transform(data[['Geography']])
ohe_encoder=onehot_encoder_geo.get_feature_names_out(['Geography'])

In [50]:
#now after encoding them, we make them into an array
## We create a matrix type
geo_encoded_df=pd.DataFrame(geo.toarray(),columns=ohe_encoder)
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [51]:
#Now we add this encoded df to the main one
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [52]:
# Creating the pickle files for the encoders so that they can be used in the future
import pickle
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo','wb') as file:
    pickle.dump(ohe_encoder,file)

In [53]:
#Find the independent and dependent features in the dataset
## in the dataset exited is the target variable, and remaining all are the features
X=data.drop(['Exited'],axis=1)
y=data['Exited']

In [54]:
#Now we give the test and train split using ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=6)

In [55]:
#scaling the data
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [56]:
X_train

array([[ 0.20043455, -1.08609344, -0.66000525, ..., -0.99707096,
        -0.58084006,  1.73575375],
       [-1.69574907, -1.08609344,  1.33062721, ...,  1.00293765,
        -0.58084006, -0.57611859],
       [-1.29164437,  0.9207311 , -0.47042121, ..., -0.99707096,
         1.72164433, -0.57611859],
       ...,
       [-0.86681634, -1.08609344, -1.51313345, ...,  1.00293765,
        -0.58084006, -0.57611859],
       [ 0.27296617, -1.08609344, -1.22875738, ..., -0.99707096,
         1.72164433, -0.57611859],
       [-0.04824527,  0.9207311 ,  1.04625114, ..., -0.99707096,
         1.72164433, -0.57611859]], shape=(7500, 12))

In [57]:
#Save the Scaler data in the pickle file
with open('scaler_data.pkl','wb') as file:
    pickle.dump(scaler,file)

# ANN Implementation

In [58]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [59]:
model=Sequential([
    Dense(70,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(35,activation='relu'),
    Dense(1,activation='sigmoid')
])

c:\Users\Hari Vallabh\GenAI\Practice\Practice\pr_env\Lib\site-packages\keras\src\layers\core\dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [60]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 70)             │           910 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 35)             │         2,485 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            36 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,431 (13.40 KB)

 Trainable params: 3,431 (13.40 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:
#Now we give the optimizer
opt=tf.keras.optimizers.Adam(learning_rate=0.005)
loss=tf.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x000001E50DA7A140>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [62]:
#To train the model, we need to complie it initially
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


In [63]:
#declare the early stopping
early_stopping=EarlyStopping(patience=15,restore_best_weights=True,monitor='val_loss')


In [64]:
# #Tensorboard
# # set the TensorBoard 
log_dir='logs/fit/'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callbacks=TensorBoard(log_dir=log_dir,histogram_freq=1)


In [65]:
#Train the model now
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=60,
    callbacks=[early_stopping,tensorboard_callbacks]
)

Epoch 1/60
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8089 - loss: 0.4422 - val_accuracy: 0.8140 - val_loss: 0.4210
Epoch 2/60
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8428 - loss: 0.3841 - val_accuracy: 0.8396 - val_loss: 0.3844
Epoch 3/60
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8555 - loss: 0.3567 - val_accuracy: 0.8496 - val_loss: 0.3675
Epoch 4/60
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8607 - loss: 0.3444 - val_accuracy: 0.8532 - val_loss: 0.3618
Epoch 5/60
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8625 - loss: 0.3386 - val_accuracy: 0.8560 - val_loss: 0.3594
Epoch 6/60
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8625 - loss: 0.3349 - val_accuracy: 0.8596 - val_loss: 0.3576
Epoch 7/60
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8636 - loss: 0.3319 - val_accuracy: 0.8592 - val_loss: 0.3563
Epoch 8/60
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8641 - loss: 0.3296 - val_accuracy: 0.

In [66]:
#save the model
model.save('practice_model_1.h5')

In [68]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 25820), started 0:00:14 ago. (Use '!kill 25820' to kill it.)